In [2]:
import pandas as pd

df = pd.read_csv('../data/online_shoppers_intention.csv')
print(df.shape)
df.head()

(12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

# Separate features and target. 'Revenue' = whether the session ended in a purchase.
X = df.drop(columns='Revenue')
y = df['Revenue']

# 80/20 split. stratify=y keeps the ~15% purchase rate identical in train and test,
# so test metrics aren't distorted by an unlucky split.
# random_state=42 makes the split reproducible across kernel restarts.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Naive baseline: always predict the majority class (no purchase).
# Any real model must beat this to justify its existence.
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)

# Baseline scores ~85% accuracy while identifying zero buyers (recall for True = 0.00).
# This shows accuracy is misleading on imbalanced data -> we will evaluate models
# on F1 / recall for the positive class instead.
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.85      1.00      0.92      2084
        True       0.00      0.00      0.00       382

    accuracy                           0.85      2466
   macro avg       0.42      0.50      0.46      2466
weighted avg       0.71      0.85      0.77      2466



c:\Users\jiahu\anaconda3\envs\mldp\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jiahu\anaconda3\envs\mldp\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jiahu\anaconda3\envs\mldp\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]

Why 85% accuracy is meaningless here: the dataset is imbalanced (~85% no-purchase vs. ~15% purchase), so a dummy model that always predicts "no purchase" scores 85% accuracy while catching literally zero actual buyers (recall = 0.00 for `True`). Accuracy just reflects the class distribution, not any real predictive skill.

What I'll use instead: F1-score (or recall) for the positive class, since missing a real buyer costs a lost sale while wrongly flagging a non-buyer only costs a wasted discount or notification. That cost asymmetry is why recall on `True` matters more than accuracy here. I'll track precision alongside it so the model isn't just flagging every session as a purchase to inflate recall; the F1 balance between the two is a much fairer signal on this imbalanced target than accuracy.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

OperatingSystems, Browser, Region, and TrafficType are stored as int64, but they're not real quantities: they're ID codes for categories (e.g. TrafficType 8 isn't "twice" TrafficType 4). Fed straight into a linear model, those numbers would imply an ordering and distance between categories that doesn't exist, so the model would learn a fake relationship from the arbitrary label numbering. They'll need one-hot encoding before modeling, the same way Month and VisitorType will.

Administrative, Informational, and ProductRelated, by contrast, are genuine counts (number of pages of that type visited in the session). Doubling the value doubles the meaning, so they stay numeric as-is.

In [4]:

print(df['TrafficType'].value_counts())

TrafficType
2     3913
1     2451
3     2052
4     1069
13     738
10     450
6      444
8      343
5      260
11     247
20     198
9       42
7       40
15      38
19      17
14      13
18      10
16       3
12       1
17       1
Name: count, dtype: int64


TrafficType has 20 levels, but the counts fall off a cliff rather than tapering: the smallest levels worth keeping sit at 247 and 198 sessions, then the next level drops straight to 42 and keeps shrinking down to single-session levels (TrafficType 12 and 17 have one session each). One-hot encoding every level would create dummy columns with almost no positive rows, and a column with one row in it carries no signal a model can learn from.

So I grouped every level below that gap into a single "Other" bin. The cut is placed at the discontinuity the data already shows (198 down to 42), not at a round number chosen for tidiness: a threshold like 250 would look neater but would swallow the two genuine levels at 247 and 198 that have plenty of data to stand on their own. Folding only the nine sparse levels below the gap leaves "Other" at 165 sessions (~1.3% of the data), small but still trainable, and no longer a set of columns with a single observation each.

In [ ]:
# The counts drop sharply from 198 to 42 sessions (see value_counts above),
# so we keep levels with at least 198 sessions and group the rest as "Other".
keep_threshold = 198
counts = df['TrafficType'].value_counts()

# Go through each session: if its TrafficType has enough sessions overall,
# keep the code (as text); otherwise label it "Other".
grouped = []
for level in df['TrafficType']:
    if counts[level] >= keep_threshold:
        grouped.append(str(level))
    else:
        grouped.append('Other')

# Add the grouped result as a new column
df['TrafficType_grouped'] = grouped

# Confirm: 11 real levels kept, sparse tail collapsed into one "Other" bin (165 sessions)
print(df['TrafficType_grouped'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# Class balance: how many sessions ended in a purchase (True) vs not (False)
revenue_counts = df['Revenue'].value_counts()
print(revenue_counts)

# Purchase rate = share of the minority (purchase) class
purchase_rate = revenue_counts[True] / df.shape[0]
print(f'Purchase rate: {purchase_rate:.1%}')

# Bar chart of the class balance
revenue_counts.plot(kind='bar')
plt.title('Class balance: purchase (True) vs no purchase (False)')
plt.xlabel('Revenue')
plt.ylabel('Number of sessions')
plt.show()

The plot confirms the imbalance: only 1,908 of 12,330 sessions end in a purchase, a 15.5% positive rate. This is the same imbalance that made accuracy misleading in the baseline cell above, and it is why the model is judged on F1 and recall for the purchase class rather than on accuracy.

In [ ]:
# Is PageValues almost deciding the outcome by itself?
# Compare its typical value for sessions that converted vs those that did not.
print('Median PageValues by outcome:')
print(df.groupby('Revenue')['PageValues'].median())
print()
print('Mean PageValues by outcome:')
print(df.groupby('Revenue')['PageValues'].mean())

# Visualise the gap in medians
median_by_outcome = df.groupby('Revenue')['PageValues'].median()
median_by_outcome.plot(kind='bar')
plt.title('Median PageValues by outcome')
plt.xlabel('Revenue')
plt.ylabel('Median PageValues')
plt.show()

PageValues is the strongest single predictor in the dataset, and that strength is exactly why it deserves a hard look. Split by outcome, the gap is extreme: for non-purchase sessions the median PageValues is 0 and the mean is about 2, while for purchase sessions the median is about 16.8 and the mean about 27. Most non-buyers sit at a PageValues of exactly 0, and buyers cluster well above it. A feature that separates the two classes this cleanly is close to deciding the label on its own, which is what invites the leakage question.

Is it leakage? PageValues is a Google Analytics metric: the average value of the pages a user visited, derived from those pages' historical transaction value. It is not literally a copy of this session's Revenue label, and in a live deployment a page's value is known as the user browses, so the feature can be observed before the outcome. On that reading it is a legitimate signal rather than future information.

The honest caveat is that PageValues is still derived from transaction behaviour, so it is a very strong proxy for the target, and its near-deterministic split should be treated with suspicion rather than celebrated. The safe handling is to be transparent that the model leans heavily on this one feature, and to sanity-check by comparing performance with and without it. If the deployment can genuinely observe PageValues before predicting, keeping it is defensible; if not, it is leakage and should be dropped.